# Lab 04: Multimodality Deep Dive (Gemini 3.1)

Gemini 3.1 was built from the ground up to be natively multimodal. In this lab, we will explore advanced scenarios for **Images**, **Audio**, and **Video** using **Gemini 3.1 Flash-Lite**.

## Objectives
1. Perform **Image Comparison** between multiple visual inputs.
2. Process audio files using the File API.
3. Implement **Temporal Reasoning** on video (extracting timestamps).

In [ ]:
import os
import time

import httpx
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Advanced Image Analysis

### 1.1 Single Image Understanding
A quick reminder of basic vision capabilities.

In [ ]:
image_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
image_data = httpx.get(image_url).content

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=[
        types.Part.from_bytes(data=image_data, mime_type="image/jpeg"),
        "Describe this image in one sentence."
    ]
)

print(f"Description: {response.text}")

### 1.2 Image Comparison
Gemini can compare multiple images in a single prompt to identify differences or common patterns.

In [ ]:
img1_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
img2_url = "https://storage.googleapis.com/generativeai-downloads/images/instrument.jpg"

img1_data = httpx.get(img1_url).content
img2_data = httpx.get(img2_url).content

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=[
        types.Part.from_bytes(data=img1_data, mime_type="image/jpeg"),
        types.Part.from_bytes(data=img2_data, mime_type="image/jpeg"),
        "What are the main differences between these two images in terms of subject "
        "and mood?"
    ]
)

display(Markdown(response.text))

## 2. Audio and Video (via File API)

### 2.1 Robust Audio Understanding
We use the File API for better reliability with audio files.

In [ ]:
audio_url = "https://storage.googleapis.com/generativeai-downloads/data/Apollo-11_Day-01-Highlights-10s.mp3"
audio_path = "sample_audio.mp3"

print("Downloading audio...")
with httpx.Client(follow_redirects=True) as http_client:
    resp = http_client.get(audio_url)
    with open(audio_path, "wb") as f:
        f.write(resp.content)

audio_file = client.files.upload(file=audio_path)

while audio_file.state.name == "PROCESSING":
    print(".", end="")
    time.sleep(2)
    audio_file = client.files.get(name=audio_file.name)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=[audio_file, "Summarize this audio clip."]
)

print(f"\nSummary: {response.text}")

# Cleanup
os.remove(audio_path)
client.files.delete(name=audio_file.name)

### 2.2 Video Temporal Reasoning
One of Gemini's strongest features is the ability to locate specific moments in a video.

In [ ]:
video_url = "https://storage.googleapis.com/generativeai-downloads/videos/Pottery.mp4"
video_path = "pottery.mp4"

print("Downloading video...")
with open(video_path, "wb") as f:
    f.write(httpx.get(video_url).content)

video_file = client.files.upload(file=video_path)

while video_file.state.name == "PROCESSING":
    print(".", end="")
    time.sleep(2)
    video_file = client.files.get(name=video_file.name)

print("\nVideo ready. Locating features...")

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=[
        video_file,
        "At what timestamp can we see the 6th item closely? Give the result in "
        "MM:SS format."
    ]
)

display(Markdown(response.text))

# Cleanup
os.remove(video_path)
client.files.delete(name=video_file.name)

## Summary

In this lab, you explored Gemini 3.1's native multimodal capabilities:
1. **Comparing multiple images** to find commonalities or differences.
2. Using the **File API for audio** processing.
3. Performing **Temporal Reasoning** on video to extract precise timestamps.